# Entrega 3 - Maestría en Ciencia de Datos - UCU
### Problema de abandono en banco - Competencia Kaggle   
#### Equipo: Enrique De Martini, Esteban Cardoso, Lucas Barrios  

## 0) Importación de librerías y definición de configuraciones iniciales

In [ ]:
# 01. IMPORTAR LIBRERIAS GENERALES ---

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import math
import joblib

In [ ]:
# 02. CONFIGURACIONES GENERALES ---

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)

## 1) Carga de datos, inspección inicial, depuración y clasificación de variables originales

In [ ]:
# 03. CARGAR DATASETS TRAIN.CSV Y TEST.CSV ---

# Cargamos dataset train.csv desde GitHub
url_train = "https://raw.githubusercontent.com/estebancardoso96/grupo_3/main/train.csv"
df = pd.read_csv(url_train)

# Cargamos dataset test.csv desde GitHub
url_test = "https://raw.githubusercontent.com/estebancardoso96/grupo_3/main/test.csv"
df_test = pd.read_csv(url_test)

# Guardamos id de test.csv para cuando debamos hacer el submission a Kaggle
test_ids = df_test["id"].copy()

In [ ]:
# 04. REALIZAR UNA INSPECCION INICIAL DEL DATASET ORIGINAL TRAIN.CSV ---

print("AUDITORIA DE VARIABLES DEL DATASET TRAIN.CSV")

print(f"\nDimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas\n")

audit = pd.DataFrame({
    'dtype':          df.dtypes,
    'nulos':          df.isnull().sum(),
    'pct_nulos':      (df.isnull().sum() / len(df) * 100).round(2),
    'valores_únicos': df.nunique(),
    'moda':           df.mode().iloc[0], # Extrae la primera fila de la moda
    'ejemplo':        df.sample(1).iloc[0], # Genera muestra aleatoria y extrae primera fila
    })
display(audit)

In [ ]:
# 05. DEPURAR DATASETS ORIGINALES TRAIN.CSV Y TEST.CSV ---

# Eliminamos columnas que no se usarán en el modelo (identificadores)
col_eliminar =['id', 'CustomerId', 'Surname']

# Depuramos los datasets eliminando las columnas no deseadas
df = df.drop(columns=col_eliminar)
df_test = df_test.drop(columns=col_eliminar)

In [ ]:
# 06. CLASIFICAR POR GRUPOS LISTA DE VARIABLES ORIGINALES A USAR EN EL MODELO ---

# Definimos variables numéricas a usar en el modelo
num_features = [
    'CreditScore',
    'Age',
    'Tenure',
    'Balance',
    'EstimatedSalary'
]

# Definimos variables categóricas a usar en el modelo
cat_features = [
    'Geography',
    'Gender',
    'NumOfProducts'
]

# Definimos variables binarias a usar en el modelo
bin_features = [
    'HasCrCard',
    'IsActiveMember'
]

# Definimos variable objetivo
target = 'Exited'

## 2) Análisis exploratorio de los datos del dataset original (EDA)

In [ ]:
# 07. REALIZAR ESTADISTICA DESCRIPTIVA DEL DATASET TRAIN.CSV DEPURADO

df.describe()

In [ ]:
# 08. REALIZAR ANALISIS DE TASA DE ABANDONO VS VARIABLES CATEGORICAS Y BINARIAS (BARPLOTS)

# Definimos las variables categóricas y binarias a analizar
variables_a_graficar = cat_features + bin_features

# Calculamos la tasa base real del dataset
tasa_base = df[target].mean()

# Configuramos el estilo estético de Seaborn
sns.set_theme(style="whitegrid")

# Creamos una grilla de 2 filas y 3 columnas
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))
axes_flat = axes.flat

for i, var in enumerate(variables_a_graficar):
    ax = axes_flat[i]
    ax.grid(False)

    # Tasa de churn y tamaño muestral por categoría
    data_grafico = (
        df.groupby(var)[target]
        .agg(tasa='mean', n='size')
        .reset_index()
    )

    orden_categorias = data_grafico[var].tolist()

    sns.barplot(
        x=var,
        y='tasa',
        data=data_grafico,
        order=orden_categorias,
        hue=var,
        hue_order=orden_categorias,
        palette="muted",
        legend=False,
        edgecolor='#404040',
        ax=ax
    )

    linea_base = ax.axhline(
        y=tasa_base,
        color='red',
        linestyle='--',
        linewidth=1.5
    )

    ax.legend(
        [linea_base],
        [f'Tasa global {tasa_base*100:.1f}%'],
        loc='upper left',
        fontsize=8.5,
        frameon=True,
        facecolor='white',
        edgecolor='none'
    )

    ax.set_title(f"Exited por {var}", fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel(var, fontsize=9)
    ax.set_ylabel("Tasa de Exited", fontsize=9)

    max_val = data_grafico['tasa'].max()
    ax.set_ylim(0, max_val + 0.12 if max_val > 0 else 0.3)

    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.tick_params(axis='y', labelsize=8.5)
    ax.grid(axis='y', linestyle=':', alpha=0.8)

    # Etiquetas: solo tamaño muestral
    for j, p in enumerate(ax.patches):
        height = p.get_height()
        if height > 0 and j < len(data_grafico):
            n_cat = int(data_grafico.iloc[j]['n'])
            ax.annotate(
                f"n={n_cat}",
                (p.get_x() + p.get_width() / 2.0, height),
                ha='center',
                va='center',
                xytext=(0, 8),
                textcoords='offset points',
                fontsize=8,
                fontweight='bold'
            )

# Apagamos cuadrantes vacíos
for j in range(len(variables_a_graficar), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle(f"Análisis de tasa de Exited vs. variables categóricas (Barplots) \n n_total={len(df)}", y=1.01, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 09. REALIZAR ANALISIS DE VARIABLES NUMERICAS VERSUS EXITED (BOXPLOTS)

# Definimos las variables numéricas a analizar
variables_boxplot = num_features

# Configuramos el estilo estético de Seaborn
sns.set_theme(style="whitegrid")

# Redefinimos el tamaño de la figura a un formato ideal (16 de base x 10 de alto)
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))

# Aplanamos la matriz de 2x3 para iterar linealmente del 0 al 5
axes_flat = axes.flat

# Mapeamos títulos amigables en español
titulos = {
    'CreditScore': 'Puntuación crediticia (CreditScore)',
    'Age': 'Edad (Age)',
    'Tenure': 'Años como cliente (Tenure)',
    'Balance': 'Saldo en cuenta (Balance)',
    'EstimatedSalary': 'Salario Estimado (EstimatedSalary)'
}

# Iteramos para construir cada Boxplot
for i, var in enumerate(variables_boxplot):
    ax = axes_flat[i]

    sns.boxplot(
        x=target,
        y=var,
        data=df,
        ax=ax,
        palette="Set2",
        hue=target,
        legend=False
    )

    # Adaptamos títulos y etiquetas al ancho de las 2 columnas
    ax.set_title(f"{var} vs Exited", fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel("¿Exited? (0=No, 1=Yes)", fontsize=9)
    ax.set_ylabel(titulos.get(var, var), fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.5)

# Apagamos cuadrantes vacíos
for j in range(len(variables_boxplot), len(axes_flat)):
    axes_flat[j].axis('off')

# Generamos título general, ajustamos layout y mostramos
plt.suptitle("Análisis de variables cuantitativas vs. Exited (Boxplots)", y=1.01, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 10. REALIZAR BARPLOTS DE TASA DE EXITED POR RANGOS DE VARIABLES NUMERICAS

# Hacemos lo mismo pero para las variables numericas, pero a estas las tendremos que dividir en rangos para los graficos de barras. 

n_cols = 2
# Calculamos las filas necesarias de forma dinámica
n_rows = math.ceil(len(num_features) / n_cols) 

# Creamos la copia de trabajo y pasamos el target a porcentaje directo (0 a 100%)
df_plot = df.copy()
df_plot['Exited_Percent'] = df_plot['Exited'].map({1: 100, 0: 0})

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))

# Aplanamos el array de ejes para poder iterar en un solo bucle limpio (aplasta una matriz 2D en 1D)
axes = axes.flatten()

# ==================================================================================================
# BUCLE EN MATRIZ DE SUBPLOTS
# ==================================================================================================
for i, var in enumerate(num_features):
    ax = axes[i]
    col_intervalo = f'{var}_Rango'
    
    # Segmentamos la variable numérica continua en 10 deciles
    # Usamos duplicates='drop' por seguridad si hay muchos valores idénticos (por lo que algunas variables no quedaran con 10 rangos)
    df_plot[col_intervalo] = pd.qcut(df_plot[var], q=20, duplicates='drop')

    data_tasa_num = (
        df_plot.groupby(col_intervalo, observed=True)['Exited_Percent']
        .mean()
        .sort_index()
        .reset_index()
    )

    data_tasa_num[col_intervalo] = data_tasa_num[col_intervalo].astype(str)
    
    # Dibujamos el barplot en el eje correspondiente
    sns.barplot(
        x=col_intervalo, 
        y='Exited_Percent', 
        data=data_tasa_num, 
        palette='Oranges_r',
        hue=col_intervalo,
        legend=False,
        ax=ax,
        edgecolor='#404040', 
        linewidth=0.8
    )
    
    # Añadimos la línea roja con la tasa base del 16% global de la empresa
    ax.axhline(y=20.04, color='crimson', linestyle='--', linewidth=1.2, label='Base Global (20.04%)' if i == 0 else "")
    
    ax.set_title(f'Tasa Exited por Rangos de {var}', fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel('Rangos', fontsize=9) 
    ax.set_ylabel('% Abandonos', fontsize=9)
    ax.set_ylim(0, max(data_tasa_num['Exited_Percent'].max() + 5, 25))
    ax.grid(axis='y', linestyle=':', alpha=0.5)

    ax.tick_params(axis='x', rotation=45)
    for tick in ax.get_xticklabels():
        tick.set_horizontalalignment('right')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])


plt.tight_layout()
plt.show()

In [ ]:
# 11. MATRIZ DE CORRELACION (EDA SOLO NUMERICAS) + RANKING TORNADO

# Correlación solo con variables numericas del modelo + target
df_eda_corr_num = df[num_features + [target]].copy()
corr_eda = df_eda_corr_num.corr(numeric_only=True)

# Reordenar para dejar Exited al final (última fila y última columna)
if target in corr_eda.columns:
    cols_ordenadas = [c for c in corr_eda.columns if c != target] + [target]
    corr_eda = corr_eda.loc[cols_ordenadas, cols_ordenadas]

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_eda,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.85},
    annot_kws={'size': 9}
 )
plt.xticks(rotation=35, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.title('Matriz de correlación (Pearson) - Solo variables numéricas', fontsize=14, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()

# Ranking de correlación con el target en modo tornado
corr_target_eda = corr_eda[target].drop(target)

# Positivas arriba y negativas abajo
ranking_pos = corr_target_eda[corr_target_eda >= 0].sort_values(ascending=False)
ranking_neg = corr_target_eda[corr_target_eda < 0].sort_values(ascending=False)
ranking_eda = pd.concat([ranking_pos, ranking_neg])

fig, ax = plt.subplots(figsize=(10, 6))
colores = ['#2ca02c' if v >= 0 else "#d62728" for v in ranking_eda.values]

ax.barh(ranking_eda.index, ranking_eda.values, color=colores, height=0.7)
ax.axvline(0, color='black', linewidth=1.0)
ax.invert_yaxis()
ax.set_xlabel('Correlacion con Exited', fontsize=11)
ax.set_title('Ranking de correlación con Exited (Tornado - Solo numéricas)', fontweight='bold', fontsize=13, pad=15)
ax.tick_params(axis='y', labelsize=10)
ax.tick_params(axis='x', labelsize=10)

for i, v in enumerate(ranking_eda.values):
    offset = 0.01 if v >= 0 else -0.01
    align = 'left' if v >= 0 else 'right'
    ax.text(v + offset, i, f'{v:+.3f}', va='center', ha=align, fontsize=9, fontweight='bold')

lim = max(abs(ranking_eda.min()), abs(ranking_eda.max())) + 0.05
ax.set_xlim(-lim, lim)
ax.grid(axis='x', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

## 3) Definición de Column Transformer y conjuntos train/test en dataset original

In [ ]:
# 12. CONSTRUIR COLUMN TRANSFORMER

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Imputamos por la mediana y escalamos las variables numéricas
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Imputamos por la moda y codificamos las variables categóricas
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

# Imputamos por la moda las variables binarias
binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

# Construimos el ColumnTransformer combinando los tres pipelines anteriores
preprocesador = ColumnTransformer(
    transformers=[
        ('escalar_num', numeric_transformer, num_features),
        ('codificar_cat', categorical_transformer, cat_features),
        ('imputar_bin', binary_transformer, bin_features),
    ],
    remainder='drop'
)

In [ ]:
# 13. SEPARAR CONJUNTO DE ENTRENAMIENTO Y TESTEO

from sklearn.model_selection import train_test_split

# Obtenemos variables predictoras y variable objetivo
X = df.drop(columns=[target])
y = df[target]

# Obtenemos dataset de entrenamiento y testeo con dataset desbalanceado
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)  # stratify=y porque dataset desbalanceado

## 4) Entrenamiento de modelo de Random Forest con dataset original

In [ ]:
# 14. ENTRENAR RANDOM FOREST CON PIPELINE Y GRIDSEARCHCV

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

pipe_rf = Pipeline(steps=[
    ('pre', preprocesador),
    ('clf', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

param_grid_rf = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features': ['sqrt', 'log2'],
    'clf__class_weight': ['balanced']
}

grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)

print(f"Mejores parámetros (RF): {grid_rf.best_params_}")
print(f"Mejor ROC-AUC (RF):      {grid_rf.best_score_:.4f}")

## 5) Evaluación de mejor modelo de Random Forest con dataset original

In [ ]:
# 15. EVALUAR EL MODELO RANDOM FOREST USANDO EL SUBSET DE TEST DE TRAIN.CSV

from sklearn.metrics import auc

y_pred = grid_rf.best_estimator_.predict(X_test)

auc = roc_auc_score(y_test, grid_rf.best_estimator_.predict_proba(X_test)[:, 1])
print(f"\n{'='*53}")
print(f"(ROC-AUC RF test = {auc:.4f})")
print(f"{'='*53}")
print(classification_report(y_test, y_pred))  

In [ ]:
# 16. CONSTRUIR MATRIZ DE CORRELACION USANDO EL SUBSET DE TEST DE TRAIN.CSV

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

best_rf_model = grid_rf.best_estimator_

y_pred_best = best_rf_model.predict(X_test)
y_proba_best = best_rf_model.predict_proba(X_test)[:, 1]

cm_best = confusion_matrix(y_test, y_pred_best)
disp_best = ConfusionMatrixDisplay(confusion_matrix=cm_best)
disp_best.plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión - Mejor Random Forest')
plt.grid(False)
plt.show()

## 5) Aplicación de técnicas de feature selection sobre dataset original

### 5a) Mutual information  
- Mutual information se basa en la entropía.  
- Mide cuánto disminuye la incertidumbre de la variable objetivo al conocer la variable predictora.  
- Detecta cualquier tipo de relación, ya sea lineal o no lineal.  
- Si el resultado es 0, las variables son estadísticamente independientes.

In [ ]:
# 17. APLICAR MUTUAL INFORMATION Y VISUALIZAR RESULTADOS

from sklearn.feature_selection import mutual_info_classif

# Calculamos Mutual Information
mi = mutual_info_classif(X_train[num_features], y_train, random_state=42)
mi_scores = pd.Series(mi, index=num_features).sort_values(ascending=False)

# Configuramos la estética del gráfico
plt.figure(figsize=(10, 6))

# Visualización gráfico de mutual information
plt.barh(
    y=mi_scores.index,
    width=mi_scores.values,
    color=sns.color_palette('magma', len(mi_scores)),
    edgecolor='black',
    linewidth=0.5
)

# Invertimos eje Y para que la variable más importante esté arriba
plt.gca().invert_yaxis()

# Generamos títulos y etiquetas
plt.title('Mutual Information Scores', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Score de Información Mutua', fontsize=11)
plt.ylabel('Variables', fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Mostramos la tabla exacta en pantalla
print("Ranking numérico de Mutual Information:")
display(mi_scores.to_frame(name='Score'))

### 5b) Permutation importance  
- El Permutation Importance es una técnica de evaluación de feature importance independiente del modelo.
- Si una variable es importante para las predicciones, al "romper" o alterar su información, el error del modelo debería aumentar drásticamente.
- El Permutation Importance se calcula sobre los datos de prueba (test set). 
- Dice qué tan importante es una variable para generalizar ante nuevos clientes, no solo para memorizar los datos de entrenamiento.

In [ ]:
# 18. APLICAR PERMUTATION IMPORTANCE CON MEJOR RANDOM FOREST y VISUALIZAR RESULTADOS 

from sklearn.inspection import permutation_importance

# Ejecutamos Permutation Importance
# Usamos el mejor modelo (best_estimator_) sobre los datos de prueba
resultado_permutacion = permutation_importance(
    estimator=grid_rf.best_estimator_,
    X=X_test,
    y=y_test,
    scoring='roc_auc',  # Usamos la misma métrica que en GridSearchCV
    n_repeats=10,       # Cantidad de veces que se mezcla cada columna para promediar
    random_state=42,
    n_jobs=-1           # Usa todos los procesadores disponibles
)

# Consolidamos los resultados en un DataFrame
df_importancia = pd.DataFrame({
    'Variable': X_test.columns,
    'Importancia_Media': resultado_permutacion.importances_mean,
    'Desviacion_Estandar': resultado_permutacion.importances_std
})

# Ordenamos las variables de mayor a menor importancia
df_importancia = df_importancia.sort_values(by='Importancia_Media', ascending=False)

# Visualizamos los resultados con un Barplot
plt.figure(figsize=(10, 6))

# Usamos matplotlib puro (plt.barh) que maneja perfectamente los errores pre-calculados (xerr)
plt.barh(
    y=df_importancia['Variable'],
    width=df_importancia['Importancia_Media'],
    xerr=df_importancia['Desviacion_Estandar'],
    capsize=4,
    color=sns.color_palette('magma', len(df_importancia)),
    edgecolor='black',
    linewidth=0.5
)

# Invertimos el eje Y para que la variable más importante quede en la parte superior
plt.gca().invert_yaxis()

plt.title('Permutation Importance (Disminución de ROC-AUC en Test)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Pérdida promedio de ROC-AUC al mezclar la variable', fontsize=11)
plt.ylabel('Variables', fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Mostrar la tabla exacta en pantalla
print("Ranking numérico de impacto:")
display(df_importancia)

### 5c) Feature importance  
- Visualiza la importancia intrínseca que el modelo de Random Forest le otorga a cada variable durante su entrenamiento.
- Lo que considera es la "Importancia basada en la impureza" (o Mean Decrease in Impurity - MDI)
- Un Random Forest está compuesto por muchos árboles de decisión.
- Cada vez que un árbol decide hacer un "corte" en una variable divide los datos en dos grupos más puros (con menos mezcla de clases).
- El algoritmo calcula cuánto "limpió" (o redujo la impureza de Gini) cada variable en cada nodo donde fue utilizada a lo largo de todos los árboles del bosque.
- La reducción de impureza se promedia para obtener un valor final.
- Si una variable redujo la incertidumbre mucho y muchas veces, tendrá una importancia alta.

In [ ]:
# 19. ANALIZAR FEATURE IMPORTANCE CON MEJOR RANDOM FOREST y VISUALIZAR RESULTADOS 

# Extraemos el modelo y el preprocesador del pipeline
modelo = grid_rf.best_estimator_.named_steps['clf']
preprocesador = grid_rf.best_estimator_.named_steps['pre']

# Obtenemos los nombres de las columnas tras el procesamiento
col_nombres = preprocesador.get_feature_names_out()

# 3. Obtenemos las importancias
importancias = modelo.feature_importances_

# Creamos el dataFrame de importancia y la ordenamos de mayor a menor
df_importancia = pd.DataFrame({'Variable': col_nombres, 'Importancia': importancias})
df_importancia = df_importancia.sort_values(by='Importancia', ascending=False)

# Graficamos (usamos head(15) para que no se vea saturado)
plt.figure(figsize=(10, 6))
sns.barplot(x='Importancia', y='Variable', data=df_importancia, palette='magma')
plt.title('Importancia de las variables (Random Forest / Dataset original)')
plt.xlabel('Importancia')
plt.ylabel('Variables')
plt.tight_layout()
plt.show()

## 6) Creación de nuevas variables que se agregarán al dataset original

Se crearán las siguientes nuevas variables para enriquecer el modelo de Random Forest:  
  
- BalancePerProduct (Balance / NumOfProducts)  
- BalancePerTenure (Balance / (Tenure + 1))  
- EstimatedSalaryToBalanceRatio (EstimatedSalary / Balance)  
- CreditScore_x_IsActiveMember (CreditScore x IsActiveMember)  
- AgeGroup (Age: <30, 30-40, 40-50, +50)  
- TenureGroup (Tenure: 0-2, 3-5, 6-10, +10)  
- CreditScoreCategory (Credit Score: <580 Pobre, 580-670 Medio, 670-740 Bueno, +740 Excelente)  
- IsSeniorCitizen (Age > 60)  
- HasBalance (Balance > 0)  
- IsHighVolumeClient (NumOfProducts > 2)  
- IsHighValueCustomer (Balance > 100000 o EstimatedSalary > 120000)
- Age_x_CreditScore (Age x CreditScore)  
- Balance_x_IsActiveMember (Balance x IsActiveMember)  
- Tenure_x_HasCrCard (Tenure x HasCrCard)  
- Geography_x_Gender (Geography + '_' + Gender)  
- LogEstimatedSalary (log(EstimatedSalary))  
- LoyaltyIndex (0.5 x ('Tenure'/10) + 0.3 x 'IsActiveMember' + 0.2 x ('NumOfProducts'/4))

Se modificará el dataset de variables originales agregando las indicadas anteriormente 

In [ ]:
# 20. CREAR NUEVAS VARIABLES Y APLICAR TRANSFORMER PARA FEATURE ENGINEERING

from sklearn.preprocessing import FunctionTransformer

def crear_nuevas_variables(X):
    X = X.copy()
    X['BalancePerProduct'] = X['Balance'] / (X['NumOfProducts'] + 0.1)
    X['BalancePerTenure'] = X['Balance'] / (X['Tenure'] + 1)
    X['EstimatedSalaryToBalanceRatio'] = X['EstimatedSalary'] / (X['Balance'] + 0.1)
    X['CreditScore_x_IsActiveMember'] = X['CreditScore'] * X['IsActiveMember']
    X['AgeGroup'] = pd.cut(
        X['Age'],
        bins=[0, 30, 40, 50, float('inf')],
        labels=['<30', '30-40', '40-50', '+50'],
        right=False
        )
    X['TenureGroup'] = pd.cut(
        X['Tenure'],
        bins=[0, 3, 6, 11, float('inf')],
        labels=['0-2', '3-5', '6-10', '+10'],
        )
    X['CreditScoreCategory'] = pd.cut(
        X['CreditScore'],
        bins=[0, 580, 670, 740, float('inf')],
        labels=['Pobre', 'Medio', 'Bueno', 'Excelente'],
        right=False
        ) # rangos definidos según indice FICO internacional
    X['IsSeniorCitizen'] = (X['Age'] > 60).astype(int)
    X['HasBalance'] = (X['Balance'] > 0).astype(int)
    X['IsHighVolumeClient'] = (X['NumOfProducts'] > 2).astype(int)
    X['IsHighValueCustomer'] = ((X['Balance'] > 100000) | (X['EstimatedSalary'] > 120000)).astype(int) # usamos sentencia OR
    X['Age_x_CreditScore'] = X['Age'] * X['CreditScore']
    X['Balance_x_IsActiveMember'] = X['Balance'] * X['IsActiveMember']
    X['Tenure_x_HasCrCard'] = X['Tenure'] * X['HasCrCard']
    X['Geography_x_Gender'] = X['Geography'].astype(str) + '_'+ X['Gender'].astype(str)
    X['LogEstimatedSalary'] = np.log1p(X['EstimatedSalary'])
    X['LoyaltyIndex'] = 0.5 * (X['Tenure'] / 10) + 0.3 * X['IsActiveMember'] + 0.2 * (X['NumOfProducts'] / 4)     
    return X

# Creamos el transformador de feature engineering usando FunctionTransformer
transformer_feature_eng = FunctionTransformer(crear_nuevas_variables)

In [ ]:
# 21. ACTUALIZAMOS LA LISTA DE VARIABLES NUMERICAS, CATEGORICAS Y BINARIAS PARA EL MODELO

# Actualizamos la lista de variables numéricas modificadas
num_features_modif = [
    'CreditScore',
    'Age',
    'Tenure',
    'Balance',
    'EstimatedSalary', 
    'BalancePerProduct',
    'BalancePerTenure',
    'EstimatedSalaryToBalanceRatio',
    'CreditScore_x_IsActiveMember',
    'Age_x_CreditScore',
    'Balance_x_IsActiveMember',
    'Tenure_x_HasCrCard',
    'LogEstimatedSalary',
    'LoyaltyIndex'
]

# Actualizamos la lista de variables categóricas modificadas
cat_features_modif = [
    'Geography',
    'Gender',
    'NumOfProducts',
    'AgeGroup',
    'TenureGroup',
    'CreditScoreCategory',
    'Geography_x_Gender'
    ]

# Actualizamos la lista de variables binarias modificadas
bin_features_modif = [
    'HasCrCard',
    'IsActiveMember',
    'IsSeniorCitizen',
    'HasBalance',
    'IsHighVolumeClient',
    'IsHighValueCustomer'    
    ]

## 7) Generar Column Transformer con nuevas variables creadas 

In [ ]:
# 22. ACTUALIZAR COLUMN TRANSFORMER CON NUEVAS VARIABLES

# Actualizamos el preprocesador con las nuevas variables
preprocesador_modif = ColumnTransformer(
    transformers=[
        ('escalar_num', numeric_transformer, num_features_modif),
        ('codificar_cat', categorical_transformer, cat_features_modif),
        ('imputar_bin', binary_transformer, bin_features_modif),
    ],
    remainder='drop'
)

## 8) Entrenamiento de modelo Random Forest con dataset modificado  

In [ ]:
# 23. ENTRENAR RANDOM FOREST CON PIPELINE Y GRIDSEARCHCV CON NUEVAS VARIABLES

# Agregamos el transformador de feature engineering al pipeline antes del preprocesador
pipe_rf_modif = Pipeline(steps=[
    ('fe', transformer_feature_eng),
    ('pre', preprocesador_modif),
    ('clf', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

param_grid_rf = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features': ['sqrt', 'log2'],
    'clf__class_weight': ['balanced']
}

grid_rf_modif = GridSearchCV(
    estimator=pipe_rf_modif,
    param_grid=param_grid_rf,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_rf_modif.fit(X_train, y_train)

print(f"Mejores parámetros (RF): {grid_rf_modif.best_params_}")
print(f"Mejor ROC-AUC (RF):      {grid_rf_modif.best_score_:.4f}")

## 9) Evaluación de modelo Random Forest modificado

In [ ]:
# 24. EVALUAR EL MODELO RANDOM FOREST USANDO EL SUBSET DE TEST DE TRAIN.CSV

from sklearn.metrics import auc

y_pred = grid_rf_modif.best_estimator_.predict(X_test)

auc = roc_auc_score(y_test, grid_rf_modif.best_estimator_.predict_proba(X_test)[:, 1])
print(f"\n{'='*53}")
print(f"(ROC-AUC RF test = {auc:.4f})")
print(f"{'='*53}")
print(classification_report(y_test, y_pred))  

In [ ]:
# 25. CONSTRUIR MATRIZ DE CORRELACION USANDO EL SUBSET DE TEST DE TRAIN.CSV

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

best_rf_model_modif = grid_rf_modif.best_estimator_

y_pred_best_modif = best_rf_model_modif.predict(X_test)
y_proba_best_modif = best_rf_model_modif.predict_proba(X_test)[:, 1]

cm_best = confusion_matrix(y_test, y_pred_best_modif)
disp_best = ConfusionMatrixDisplay(confusion_matrix=cm_best)
disp_best.plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión - Mejor Random Forest')
plt.grid(False)
plt.show()

## 10) Aplicación de técnicas de feature selection sobre dataset modificado

### 10a) Mutual information

In [ ]:
# 26. APLICAR MUTUAL INFORMATION SOBRE DATASET MODIFICADO Y VISUALIZAR RESULTADOS

# Modificamos el dataset X_train para incluir las nuevas variables creadas
X_train_modif = crear_nuevas_variables(X_train)

# Calculamos Mutual Information
mi = mutual_info_classif(X_train_modif[num_features_modif], y_train, random_state=42)
mi_scores = pd.Series(mi, index=num_features_modif).sort_values(ascending=False)

# Configuramos la estética del gráfico
plt.figure(figsize=(10, 6))

# Visualización gráfico de mutual information
plt.barh(
    y=mi_scores.index,
    width=mi_scores.values,
    color=sns.color_palette('magma', len(mi_scores)),
    edgecolor='black',
    linewidth=0.5
)

# Invertimos eje Y para que la variable más importante esté arriba
plt.gca().invert_yaxis()

# Generamos títulos y etiquetas
plt.title('Mutual Information Scores', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Score de Información Mutua', fontsize=11)
plt.ylabel('Variables', fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Mostramos la tabla exacta en pantalla
print("Ranking numérico de Mutual Information:")
display(mi_scores.to_frame(name='Score'))

### 10b) Permutation importance

In [ ]:
# 27. APLICAR PERMUTATION IMPORTANCE CON MEJOR RANDOM FOREST SOBRE DATASET MODIFICADO y VISUALIZAR RESULTADOS 

# Modificamos el dataset X_test para incluir las nuevas variables creadas
X_test_modif = crear_nuevas_variables(X_test)

# Ejecutamos Permutation Importance
# Usamos el mejor modelo (best_estimator_) sobre los datos de prueba
resultado_permutacion = permutation_importance(
    estimator=grid_rf_modif.best_estimator_,
    X=X_test_modif,
    y=y_test,
    scoring='roc_auc',  # Usamos la misma métrica que en GridSearchCV
    n_repeats=10,       # Cantidad de veces que se mezcla cada columna para promediar
    random_state=42,
    n_jobs=-1           # Usa todos los procesadores disponibles
)

# Consolidamos los resultados en un DataFrame
df_importancia = pd.DataFrame({
    'Variable': X_test_modif.columns,
    'Importancia_Media': resultado_permutacion.importances_mean,
    'Desviacion_Estandar': resultado_permutacion.importances_std
})

# Ordenamos las variables de mayor a menor importancia
df_importancia = df_importancia.sort_values(by='Importancia_Media', ascending=False)

# Visualizamos los resultados con un Barplot
plt.figure(figsize=(10, 10))

# Usamos matplotlib puro (plt.barh) que maneja perfectamente los errores pre-calculados (xerr)
plt.barh(
    y=df_importancia['Variable'],
    width=df_importancia['Importancia_Media'],
    xerr=df_importancia['Desviacion_Estandar'],
    capsize=4,
    color=sns.color_palette('magma', len(df_importancia)),
    edgecolor='black',
    linewidth=0.5
)

# Invertimos el eje Y para que la variable más importante quede en la parte superior
plt.gca().invert_yaxis()

plt.title('Permutation Importance (Disminución de ROC-AUC en Test)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Pérdida promedio de ROC-AUC al mezclar la variable', fontsize=11)
plt.ylabel('Variables', fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Mostrar la tabla exacta en pantalla
print("Ranking numérico de impacto:")
display(df_importancia)

### 10c) Feature importance

In [ ]:
# 28. ANALIZAR FEATURE IMPORTANCE CON MEJOR RANDOM FOREST SOBRE DATASET MODIFICADO y VISUALIZAR RESULTADOS 

# Extraemos el modelo y el preprocesador del pipeline
modelo = grid_rf_modif.best_estimator_.named_steps['clf']
preprocesador = grid_rf_modif.best_estimator_.named_steps['pre']

# Obtenemos los nombres de las columnas tras el procesamiento
col_nombres = preprocesador.get_feature_names_out()

# 3. Obtenemos las importancias
importancias = modelo.feature_importances_

# Creamos el dataFrame de importancia y la ordenamos de mayor a menor
df_importancia = pd.DataFrame({'Variable': col_nombres, 'Importancia': importancias})
df_importancia = df_importancia.sort_values(by='Importancia', ascending=False)

# Graficamos (usamos head(15) para que no se vea saturado)
plt.figure(figsize=(10, 10))
sns.barplot(x='Importancia', y='Variable', data=df_importancia, palette='magma')
plt.title('Importancia de las variables (Random Forest / Dataset modificado)')
plt.xlabel('Importancia')
plt.ylabel('Variables')
plt.tight_layout()
plt.show()

### 10d) Análisis Recursive Feature Elimination (RFE)

In [ ]:
# 29. ANÁLISIS DE RFE (Recursive Feature Elimination con CV) ---

from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier

print("Iniciando análisis RFECV (esto puede tomar unos minutos)...")

# Definimos el estimador base usando los mejores hiperparámetros que ya encontraste en tu GridSearch
best_params = grid_rf_modif.best_params_
# Limpiamos los prefijos 'clf__' para pasarlos directamente al clasificador
clean_params = {k.replace('clf__', ''): v for k, v in best_params.items()}
base_clf = RandomForestClassifier(random_state=42, **clean_params)

# Creamos un pipeline intermedio que aplique el feature engineering y el preprocesamiento 
# para extraer las matrices transformadas y pasarlas a RFE
from sklearn.pipeline import Pipeline as SkPipeline
preprocessing_pipeline = SkPipeline(steps=[
    ('fe', grid_rf_modif.best_estimator_.named_steps['fe']),
    ('pre', grid_rf_modif.best_estimator_.named_steps['pre'])
])

# Transformamos X_train y X_test para trabajar directamente con las features procesadas (One-Hot, Scaled, etc.)
X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

# Recuperamos los nombres finales de las columnas post-preprocesamiento
feature_names_final = grid_rf_modif.best_estimator_.named_steps['pre'].get_feature_names_out()

# Configuramos RFECV utilizando la métrica ROC-AUC y validación cruzada de 5 folds
rfecv = RFECV(
    estimator=base_clf,
    step=1,                # Elimina de 1 en 1 variable por iteración (o pon ej. 0.05 para ir más rápido)
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Ajustamos RFECV con los datos procesados
rfecv.fit(X_train_processed, y_train)

print(f"\nNúmero óptimo de características a mantener: {rfecv.n_features_}")

# --- VISUALIZACIÓN GRÁFICA DE RFECV ---
plt.figure(figsize=(10, 6))
plt.title('Rendimiento del Modelo (ROC-AUC) vs. Número de Características', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Número de características seleccionadas', fontsize=11)
plt.ylabel('Score de Validación Cruzada (ROC-AUC)', fontsize=11)

# Graficamos la curva de desempeño con su desviación estándar
plt.plot(
    range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
    rfecv.cv_results_['mean_test_score'],
    color='b',
    linewidth=2,
    marker='o',
    markersize=3
)

# Marcamos con una línea vertical el punto óptimo
plt.axvline(
    x=rfecv.n_features_, 
    color='r', 
    linestyle='--', 
    label=f'Óptimo de variables: {rfecv.n_features_}'
)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# --- RANKING Y SELECCIÓN DE VARIABLES ---
# Creamos un DataFrame con el ranking de todas las variables
df_rfe_ranking = pd.DataFrame({
    'Variable': feature_names_final,
    'Seleccionada': rfecv.support_,
    'Ranking': rfecv.ranking_
})

# Ordenamos por ranking (1 significa que fueron seleccionadas como las mejores)
df_rfe_ranking = df_rfe_ranking.sort_values(by='Ranking', ascending=True)

print("\nRanking de variables según RFE:")
display(df_rfe_ranking)

# Variables que el modelo sugiere eliminar (Ranking > 1)
variables_a_eliminar = df_rfe_ranking[df_rfe_ranking['Seleccionada'] == False]['Variable'].tolist()
print(f"\nCantidad de variables descartadas por RFE: {len(variables_a_eliminar)}")

## 11) Conclusiones del Feature Engineering aplicado

Al aplicar Feature Engineering en el modelo Random Forest, pasando del dataset original a uno modificado agregando variables observamos que :  
  
- ROC AUC RF pasó de 0.9246 a 0.9270 lo cual implica una mejora  
- Según Mutual information:  
    - "Age" es la variable con mayor dependencia con el target  
    - Le siguen en el ranking la nueva variable "BalancePerProduct" y la variable original "Balance".
    - El modelo modificado captura patrones observados en el EDA  
- Según Permutation Importance:  
    - Las variables originales "NumOfProducts", "Age" y "Balance" siguen siendo importantes.
    - Las nuevas variables creadas muestran una importancia menor en esta métrica.    
- Según Feature importance:  
    - La nueva variable creada "AgeGroup" se encuentra en el top 3 del ranking   
    - La nueva variable creada "BalancePerProduct" se ha posicionado alto en el ranking por encima de variables originales 
    - La nueva variable "BalancePerTenure" se encuentra en la parte superior media  
    - Las variables originales "NumOfProducts" y "Age" siguen siendo las más importantes  
    - No se aprecia que otras variables originales que antes eran importantes hayan dejado de serlo 
- Concluimos que el Feature Engineering está aportando en la mejora del modelo Random Forest

## 12) Generación de archivo para submission en Kaggle

In [ ]:
# 29. GENERAR ARCHIVO DE SUBMISSION PARA KAGGLE

# Aseguramos que el pipeline (que incluye el feature engineering) prediga sobre df_test
# Como pipe_rf_modif incluye 'fe' (la función de ingeniería) y 'pre' (preprocesador),
# los datos de df_test serán transformados automáticamente igual que los de entrenamiento.
y_test_pred_proba = grid_rf_modif.best_estimator_.predict_proba(df_test)[:, 1]

# Creamos el DataFrame de submission
# Kaggle solicita dos columnas: 'id' y 'Exited'
submission = pd.DataFrame({
    'id': test_ids,
    'Exited': y_test_pred_proba
})

# Guardamos el archivo
submission.to_csv('submission_modelo_modificado_rf.csv', index=False)

print("¡Archivo 'submission_modelo_modificado_rf.csv' generado con éxito!")
display(submission.head())